# So sánh truy hồi trên toàn bộ tập dev: BM25 / vector / hybrid × rerank × lọc phiên bản

Chạy trên **máy local** bằng venv của dự án (kernel `venv/bin/python`). **Embedding và reranker nằm trên model server Colab**
(`scripts/setup_server_v2.sh`), máy local không nạp model nào và không dùng GPU.

**Trước khi chạy:** trên Colab chạy `setup_server_v2.sh`, rồi chép `MODEL_SERVER_URL` (và `OLLAMA_BASE_URL`) vào `.env` ở thư mục
gốc repo. Phần so sánh truy hồi không cần LLM; Ollama chỉ được kiểm tra để ghi vào manifest.

Notebook đo **12 cấu hình** {BM25, vector, hybrid} × {không/có rerank} × {không/có lọc phiên bản} trên các bộ dev (bộ 01, bộ 02,
dev đợt 1–4):
- kết quả **tách theo từng bộ**, không gộp (nhận xét ý 2), kèm khoảng tin cậy bootstrap 95% và McNemar;
- file kết quả ghi commit, ngày, URL model server, phiên bản thư viện để làm minh chứng (nhận xét ý 6);
- **không chạy tập kiểm chứng** (`test_kiem_chung_*.json`), tập đó chỉ chạy một lần sau khi đóng băng cấu hình;
- thời gian mỗi câu **gồm độ trễ mạng qua Cloudflare tunnel**, không so trực tiếp với số đo tại chỗ.

Khoảng 2 700 lần gọi model server (12 cấu hình × 160 câu dev), mất cỡ 35–50 phút tùy tunnel; giữ tab Colab mở. Tunnel lỗi tạm thời thì `src/remote_models.py` tự gọi lại.


In [1]:
# ---- Tham số ----
RETRIEVERS = ["bm25", "vector", "hybrid"]
CAU_HINH_CHOT = "vector+rerank+loc"  # cấu hình đang chốt trong báo cáo (Chương 8)

import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "src" / "config.py").exists(), f"Không thấy src/config.py dưới {ROOT} — mở notebook từ thư mục repo"
os.chdir(ROOT)
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # máy local không dùng GPU: model chạy trên Colab

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)  # override: lấy URL mới nhất trong .env dù kernel đã đặt URL cũ
os.environ["RAG_PROFILE"] = "server"
for k in ("RAG_RERANK", "RAG_LOC_PHIEN_BAN", "RAG_SEED", "RAG_TEMPERATURE"):
    os.environ.pop(k, None)  # không để biến môi trường ép cấu hình
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "eval")]

def git(*args):
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True).stdout.strip()

COMMIT, BRANCH = git("rev-parse", "--short", "HEAD"), git("branch", "--show-current")
from dong_bang import chua_commit as _chua_commit
chua_commit = _chua_commit()  # mọi thay đổi chưa commit trong src/, eval/ (trừ eval/results), data/processed
print(f"repo {ROOT} | nhánh {BRANCH} | commit {COMMIT}")
if chua_commit:
    print("CẢNH BÁO: code/dữ liệu đo có thay đổi chưa commit — số liệu không ứng với commit trên:\n" + chua_commit)


repo /mnt/data/1_Projects/sgu-rag-llamaindex | nhánh main | commit a6e63eb
CẢNH BÁO: code/dữ liệu đo có thay đổi chưa commit — số liệu không ứng với commit trên:
M eval/evaluate.py
 M eval/luat_cham.py
 M eval/tra_loi_dev.py
?? eval/bao_phu.py
?? eval/dev_questions_dot4.json
?? eval/dong_bang.py
?? eval/kiem_tra_trung.py


## 1. Kiểm tra model server (embedding + reranker) và Ollama trên Colab


In [2]:
import httpx
from remote_models import goi_server

MODEL_SERVER_URL = os.environ.get("MODEL_SERVER_URL", "").strip()
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "").strip()
if not MODEL_SERVER_URL:
    raise SystemExit("MODEL_SERVER_URL trong .env đang trống: chạy setup_server_v2.sh trên Colab rồi chép URL vào .env")
SERVER_HEALTH = goi_server("GET", f"{MODEL_SERVER_URL}/health", timeout=30).json()
print("model server:", MODEL_SERVER_URL, "\n ", SERVER_HEALTH)
try:
    httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=15).raise_for_status()
    print("Ollama:", OLLAMA_BASE_URL, "sẵn sàng (không dùng trong phần so sánh truy hồi)")
except (httpx.HTTPError, ValueError) as e:
    print("Ollama không gọi được (không ảnh hưởng phần truy hồi):", type(e).__name__)


model server: https://fans-compete-lang-himself.trycloudflare.com 
  {'embed_model': 'AITeamVN/Vietnamese_Embedding', 'reranker_model': 'AITeamVN/Vietnamese_Reranker', 'reranker_max_length': 2048, 'device': 'cuda', 'fp16': True}
Ollama: https://arch-stats-oil-river.trycloudflare.com sẵn sàng (không dùng trong phần so sánh truy hồi)


## 2. Nạp index (profile `server`, embedding câu hỏi qua model server)


In [3]:
import logging
logging.basicConfig(level=logging.WARNING)
from config import get_config
from index_builder import build_or_load_index

cfg0 = get_config()
assert cfg0.model_server_url, "cấu hình chưa nhận MODEL_SERVER_URL — sẽ nạp model tại chỗ"
# src/config.py đọc URL một lần lúc import: kernel đã import trước khi sửa .env thì vẫn gọi tunnel cũ (lỗi 530)
if cfg0.model_server_url != MODEL_SERVER_URL:
    raise SystemExit(f"Kernel đang giữ URL cũ ({cfg0.model_server_url}) khác .env ({MODEL_SERVER_URL}): "
                     "Kernel > Restart Kernel rồi chạy lại từ đầu")
index = build_or_load_index(cfg0)  # kiểm tra server đúng model + reranker 2048 token; index cũ khớp fingerprint thì nạp lại
nodes = list(index.docstore.docs.values())
so_van_ban = len({n.metadata.get("file") for n in nodes})
print(f"{len(nodes)} Node từ {so_van_ban} văn bản | embedding {cfg0.embed_model_name} | top_k={cfg0.similarity_top_k}")
if len(nodes) != 118:
    print("CẢNH BÁO: số Node khác 118 trong báo cáo — kiểm tra corpus/nhánh trước khi dùng số liệu")


/mnt/data/1_Projects/sgu-rag-llamaindex/venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


118 Node từ 20 văn bản | embedding AITeamVN/Vietnamese_Embedding | top_k=10


## 3. Chạy 12 cấu hình trên các bộ dev

- `bm25`: tìm từ khóa thuần (`src/bm25.py`), top-10.
- `vector`: embedding top-10.
- `hybrid`: vector + BM25 trộn bằng `QueryFusionRetriever` (RRF), top-10.
- `+rerank`: `AITeamVN/Vietnamese_Reranker` đọc 2 048 token, **giữ 3 Node** (đúng số Node đưa vào LLM): với các cấu hình có
  rerank chỉ R@1, R@3 và MRR so sánh được; R@5/R@10 bị chặn ở 3.
- `+loc`: lọc phiên bản (`src/phien_ban.py`): bỏ văn bản hết hiệu lực, lọc theo khóa, đưa bản sửa đổi lên trước.


In [4]:
import dataclasses, time
from bm25 import BM25Retriever
from evaluate import BO_CAU_HOI, danh_gia
from retriever import get_postprocessors, get_retriever, search

def ten_cau_hinh(r, rr, loc):
    return r + ("+rerank" if rr else "") + ("+loc" if loc else "")

MA_TRAN = [(r, rr, loc) for r in RETRIEVERS for rr in (False, True) for loc in (False, True)]
BO = [f for f, _ in BO_CAU_HOI]
ket_qua = {}   # (cấu hình, bộ) -> KetQua
mo_ta_cau_hinh = {}
t_bat_dau = time.time()
for r, rr, loc in MA_TRAN:
    ten = ten_cau_hinh(r, rr, loc)
    cfg = dataclasses.replace(cfg0, use_hybrid=(r == "hybrid"), use_rerank=rr, loc_phien_ban=loc)
    if r == "bm25":
        retriever = BM25Retriever(nodes=nodes, similarity_top_k=cfg.similarity_top_k, k1=cfg.bm25_k1, b=cfg.bm25_b)
    else:
        retriever = get_retriever(index, cfg)
    posts = get_postprocessors(cfg, index)
    tim = lambda q, r_=retriever, p_=posts, c_=cfg: search(index, c_, q, postprocessors=p_, retriever=r_)
    tim("khởi động")  # lần gọi đầu chậm hơn, không tính giờ
    mo_ta_cau_hinh[ten] = {**cfg.mo_ta(), "retriever": r}
    for ten_file, dung in BO_CAU_HOI:
        ket_qua[(ten, ten_file)] = danh_gia(ten_file, dung, tim, ten, mo_ta_cau_hinh[ten])
    hang = "  ".join(f"{f.removesuffix('.json')[-5:]} {ket_qua[(ten, f)].recall(1):4.0%}" for f in BO)
    print(f"{ten:22} R@1: {hang}", flush=True)
print(f"Xong {len(MA_TRAN)} cấu hình trong {time.time() - t_bat_dau:.0f} s")


bm25                   R@1: ns_01  58%  ns_02  40%  _dot1  80%  _dot2  60%  _dot3  73%  _dot4  59%
bm25+loc               R@1: ns_01  58%  ns_02  70%  _dot1  70%  _dot2  53%  _dot3  87%  _dot4  59%


KeyboardInterrupt: 

## 4. Bảng kết quả (tách theo bộ, không gộp)


In [ ]:
from IPython.display import Markdown, display
from metrics import mcnemar

TEN_BO = {"test_questions_01.json": "Bộ 01 (QC 2021)", "test_questions_02.json": "Bộ 02 (QĐ 3183 sửa đổi)",
          "dev_questions_dot1.json": "Dev đợt 1", "dev_questions_dot2.json": "Dev đợt 2 (tiếng Anh)",
          "dev_questions_dot3.json": "Dev đợt 3", "dev_questions_dot4.json": "Dev đợt 4"}
CAU_HINH = [ten_cau_hinh(*m) for m in MA_TRAN]

def bang_tong_quan():
    dong = ["| Cấu hình | " + " | ".join(f"{TEN_BO.get(f, f)} ({ket_qua[(CAU_HINH[0], f)].so_cau})" for f in BO) + " |",
            "|---|" + "---|" * len(BO)]
    for ten in CAU_HINH:
        o = [f"{sum(ket_qua[(ten, f)].hit(1))}/{ket_qua[(ten, f)].so_cau}" for f in BO]
        dong.append(f"| {'**' + ten + '**' if ten == CAU_HINH_CHOT else ten} | " + " | ".join(o) + " |")
    return "\n".join(dong)

def bang_mot_bo(f):
    dong = [f"#### {TEN_BO.get(f, f)} — {ket_qua[(CAU_HINH[0], f)].so_cau} câu",
            "| Cấu hình | R@1 [CI 95%] | R@3 | MRR | ms/câu (trung vị) | McNemar R@1 so với vector | so với " + CAU_HINH_CHOT + " |",
            "|---|---|---|---|---|---|---|"]
    for ten in CAU_HINH:
        kq = ket_qua[(ten, f)]
        _, lo, hi = kq.ci(1)
        ms = sorted(kq.thoi_gian_ms)[len(kq.thoi_gian_ms) // 2]
        cot = []
        for goc in ("vector", CAU_HINH_CHOT):
            if ten == goc:
                cot.append("—")
                continue
            b, c, p = mcnemar(ket_qua[(goc, f)].hit(1), kq.hit(1))
            cot.append(f"−{b}/+{c}, p={p:.3f}")
        dong.append(f"| {ten} | {kq.recall(1):.1%} [{lo:.0%}–{hi:.0%}] | {kq.recall(3):.1%} | {kq.mrr():.3f} | {ms:.0f} | "
                    + " | ".join(cot) + " |")
    return "\n".join(dong)

GHI_CHU = (
    "> Số câu đúng ở top-1. Luật chấm: bộ 01/02 theo cặp (số hiệu văn bản, Điều); các đợt dev theo tiền tố file "
    "(và Điều nếu nhãn có Điều), nên văn bản/Điều bị cắt thành nhiều Node được tính đúng khi lấy được **bất kỳ** mảnh nào "
    "(14/30 câu của đợt 2–3; đợt 4 gán nhãn đến từng Điều, kể cả `bh:`/`sd:`) — không so trực tiếp giữa các bộ. "
    "Cấu hình có rerank giữ 3 Node. "
    "McNemar: −b = số câu chỉ cấu hình gốc đúng, +c = số câu chỉ cấu hình ở dòng đó đúng. ms/câu gồm độ trễ mạng tới "
    "model server Colab.")
CAU_HINH_CHINH = ["bm25", "vector", "vector+loc", "vector+rerank", CAU_HINH_CHOT, "hybrid+rerank+loc"]

def bang_theo_loai():
    """R@1 theo loại câu (trường `loai`, vd `khoa` = đáp án phụ thuộc khóa tuyển sinh) cho các bộ có trường này."""
    from evaluate import doc_cau_hoi
    theo_loai = {}  # loại -> cấu hình -> [đúng top-1]
    for f in BO:
        loai = [q.get("loai") for q in doc_cau_hoi(f)]
        if not any(loai):
            continue
        for ten in CAU_HINH_CHINH:
            for l, h in zip(loai, ket_qua[(ten, f)].hit(1)):
                theo_loai.setdefault(l or "-", {}).setdefault(ten, []).append(h)
    if not theo_loai:
        return "_Không bộ dev nào có trường `loai`._"
    dong = ["| Loại câu | Số câu | " + " | ".join(CAU_HINH_CHINH) + " |", "|---|---|" + "---|" * len(CAU_HINH_CHINH)]
    for l, theo_ch in sorted(theo_loai.items()):
        n = len(theo_ch[CAU_HINH_CHINH[0]])
        dong.append(f"| {l} | {n} | " + " | ".join(f"{sum(theo_ch[t])}/{n}" for t in CAU_HINH_CHINH) + " |")
    return "\n".join(dong)

MD_KET_QUA = "\n\n".join(["### Tổng quan R@1 (số câu đúng / số câu)", bang_tong_quan(), GHI_CHU,
                          "### R@1 theo loại câu (các bộ có trường `loai`)", bang_theo_loai(),
                          "### Chi tiết từng bộ", *[bang_mot_bo(f) for f in BO]])
display(Markdown(MD_KET_QUA))


## 5. Câu trượt top-3 của cấu hình chốt (để phân tích lỗi, Mục 11.3)


In [ ]:
for f in BO:
    kq = ket_qua[(CAU_HINH_CHOT, f)]
    if kq.ngoai_top3:
        print(f"== {TEN_BO.get(f, f)}")
        for q, d, top3 in kq.ngoai_top3:
            print(f"  - {q}\n    đúng: {d}\n    top-3: {top3}")


## 6. Lưu kết quả và minh chứng (JSON + Markdown + manifest) vào `eval/results/`


In [ ]:
import importlib.metadata as im, json, platform

ngay = time.strftime("%Y-%m-%d")
# Có giờ trong tên để không ghi đè lần chạy trước cùng ngày, cùng commit (minh chứng cũ phải giữ nguyên)
OUT = ROOT / "eval" / "results" / f"so-sanh-truy-hoi-{ngay}-{time.strftime('%H%M')}-{COMMIT}{'-chua-commit' if chua_commit else ''}"
OUT.mkdir(parents=True, exist_ok=False)
manifest = {
    "thoi_diem": time.strftime("%Y-%m-%d %H:%M:%S"),
    "lenh": "notebooks/so_sanh_truy_hoi_dev.ipynb",
    "nhanh": BRANCH, "commit": COMMIT, "co_thay_doi_chua_commit": bool(chua_commit),
    "model_server": {"url": MODEL_SERVER_URL, "health": SERVER_HEALTH},
    "may_local": {"python": platform.python_version(),
                  **{p: im.version(p) for p in ("llama-index-core", "llama-index-embeddings-huggingface", "nltk")}},
    "corpus": {"so_node": len(nodes), "so_van_ban": so_van_ban},
    "bo_cau_hoi": {f: ket_qua[(CAU_HINH[0], f)].so_cau for f in BO},
    "cau_hinh_chot": CAU_HINH_CHOT,
    "cau_hinh": mo_ta_cau_hinh,
}
(OUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
(OUT / "ket-qua.json").write_text(json.dumps([kq.to_dict() for kq in ket_qua.values()], ensure_ascii=False, indent=1),
                                  encoding="utf-8")
(OUT / "ket-qua.md").write_text(
    f"# So sánh truy hồi trên tập dev — {manifest['thoi_diem']}, commit `{COMMIT}`\n\n"
    f"{len(nodes)} Node / {so_van_ban} văn bản, profile server. Embedding và reranker fp16 trên model server Colab "
    f"({SERVER_HEALTH.get('device')}), gọi qua Cloudflare tunnel.\n\n" + MD_KET_QUA + "\n", encoding="utf-8")
print("Đã ghi", OUT)
